In [45]:
!pip install pyspark

In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [47]:
spark = SparkSession.builder \
    .appName("SQL to PySpark Phase 2") \
    .getOrCreate()

In [48]:
import pandas as pd

data = [
    [1001,1,"P201","2026-01-05",2,250,10,"UPI","Delivered"],
    [1002,2,"P203","2026-01-06",1,1200,0,"Credit Card","Delivered"],
    [1003,3,"P202","2026-01-06",3,450,25,"Debit Card","Shipped"],
    [1004,4,"P204","2026-01-07",1,800,50,"Cash","Cancelled"],
    [1005,1,"P205","2026-01-08",2,350,20,"UPI","Delivered"],
    [1006,5,"P201","2026-01-08",1,250,0,"Credit Card","Delivered"],
    [1007,6,"P206","2026-01-09",4,150,15,"UPI","Delivered"],
    [1008,7,"P207","2026-01-09",2,500,30,"Debit Card","Shipped"],
    [1009,8,"P208","2026-01-10",1,900,0,"Cash","Delivered"],
    [1010,2,"P209","2026-01-10",2,300,10,"UPI","Delivered"],
    [1011,9,"P210","2026-01-11",1,650,20,"Credit Card","Delivered"],
    [1012,10,"P211","2026-01-11",5,120,0,"UPI","Delivered"],
    [1013,3,"P212","2026-01-12",2,400,15,"Debit Card","Delivered"],
    [1014,4,"P213","2026-01-12",3,275,25,"Credit Card","Returned"],
    [1015,5,"P214","2026-01-13",1,1500,100,"UPI","Delivered"],
]

columns = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "quantity",
    "unit_price",
    "discount",
    "payment_mode",
    "status"
]

df = pd.DataFrame(data, columns=columns)

df.to_csv("orders.csv", index=False)

print("orders.csv created successfully!")
print(df.head())

orders.csv created successfully!
   order_id  customer_id product_id  order_date  quantity  unit_price  \
0      1001            1       P201  2026-01-05         2         250   
1      1002            2       P203  2026-01-06         1        1200   
2      1003            3       P202  2026-01-06         3         450   
3      1004            4       P204  2026-01-07         1         800   
4      1005            1       P205  2026-01-08         2         350   

   discount payment_mode     status  
0        10          UPI  Delivered  
1         0  Credit Card  Delivered  
2        25   Debit Card    Shipped  
3        50         Cash  Cancelled  
4        20          UPI  Delivered  


In [49]:
from google.colab import files
files.download("orders.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
from google.colab import files

uploaded = files.upload()

Saving customers.csv to customers (3).csv
Saving orders (1).csv to orders (1) (2).csv


In [61]:
customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("customers.csv")

In [62]:
from pyspark.sql.functions import split, regexp_replace, col

# Read the file as text
raw = spark.read.text("orders.csv")

# Remove double quotes
raw = raw.withColumn(
    "value",
    regexp_replace(col("value"), '"', "")
)

# Split into columns
orders = raw.select(
    split(col("value"), ",").getItem(0).alias("order_id"),
    split(col("value"), ",").getItem(1).alias("customer_id"),
    split(col("value"), ",").getItem(2).alias("product_id"),
    split(col("value"), ",").getItem(3).alias("order_date"),
    split(col("value"), ",").getItem(4).alias("quantity"),
    split(col("value"), ",").getItem(5).alias("unit_price"),
    split(col("value"), ",").getItem(6).alias("discount"),
    split(col("value"), ",").getItem(7).alias("payment_mode"),
    split(col("value"), ",").getItem(8).alias("status")
)

# Remove header row
orders = orders.filter(col("order_id") != "order_id")

orders.show(5)
orders.printSchema()

+--------+-----------+----------+----------+--------+----------+--------+------------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount|payment_mode|   status|
+--------+-----------+----------+----------+--------+----------+--------+------------+---------+
|    1001|          1|      P201|2026-01-05|       2|       250|      10|         UPI|Delivered|
|    1002|          2|      P203|2026-01-06|       1|      1200|       0| Credit Card|Delivered|
|    1003|          3|      P202|2026-01-06|       3|       450|      25|  Debit Card|  Shipped|
|    1004|          4|      P204|2026-01-07|       1|       800|      50|        Cash|Cancelled|
|    1005|          1|      P205|2026-01-08|       2|       350|      20|         UPI|Delivered|
+--------+-----------+----------+----------+--------+----------+--------+------------+---------+
only showing top 5 rows
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id

In [63]:
orders = orders \
    .withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .withColumn("discount", col("discount").cast("double"))

Mini Cleaning Task

In [64]:
orders = orders.withColumn(
    "order_amount",
    col("quantity") * col("unit_price") - col("discount")
)

In [65]:
customers.show(5)

orders.show(5)

customers.printSchema()

orders.printSchema()

+-----------+----------+---------+--------------------+------------+------------+-----------+-----+--------+
|customer_id|first_name|last_name|               email|phone_number|     address|       city|state|zip_code|
+-----------+----------+---------+--------------------+------------+------------+-----------+-----+--------+
|          1|      John|    Smith|john.smith@domain...|    555-0001|  123 Elm St|Springfield|   IL|   62701|
|          2|      Emma|    Jones|emma.jones@webmai...|    555-0002|  456 Oak St|Centerville|   OH|   45459|
|          3|    Olivia|    Brown|olivia.brown@outl...|    555-0003| 789 Pine St| Greenville|   SC|   29601|
|          4|      Liam|  Johnson|liam.johnson@gmai...|    555-0004|101 Maple St|  Riverside|   CA|   92501|
|          5|      Noah| Williams|noah.williams@yah...|    555-0005|202 Birch St|   Lakeside|   TX|   75001|
+-----------+----------+---------+--------------------+------------+------------+-----------+-----+--------+
only showing top 5 

Exercise 1: Total Order Amount for Each Customer
#SQL

In [66]:
spark.sql("""
SELECT
    customer_id,
    SUM(CAST(quantity AS INT) * CAST(unit_price AS DOUBLE) - CAST(discount AS DOUBLE)) AS total_spend
FROM orders
GROUP BY customer_id
ORDER BY total_spend DESC
""").show()

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          3|     2110.0|
|          2|     1790.0|
|          5|     1650.0|
|          4|     1550.0|
|          1|     1170.0|
|          7|      970.0|
|          8|      900.0|
|          9|      630.0|
|         10|      600.0|
|          6|      585.0|
+-----------+-----------+



pyspark

In [67]:
orders = orders.withColumn(
    "order_amount",
    col("quantity").cast("int") *
    col("unit_price").cast("double") -
    col("discount").cast("double")
)

customer_spend = orders.groupBy("customer_id") \
    .agg(sum("order_amount").alias("total_spend")) \
    .orderBy(col("total_spend").desc())

customer_spend.show()

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          3|     2110.0|
|          2|     1790.0|
|          5|     1650.0|
|          4|     1550.0|
|          1|     1170.0|
|          7|      970.0|
|          8|      900.0|
|          9|      630.0|
|         10|      600.0|
|          6|      585.0|
+-----------+-----------+



Exercise 2: Top 3 Customers by Total Spend

SQL

In [71]:
spark.sql("""
SELECT
    customer_id,
    SUM(CAST(quantity AS INT) * CAST(unit_price AS DOUBLE) - CAST(discount AS DOUBLE)) AS total_spend
FROM orders
GROUP BY customer_id
ORDER BY total_spend DESC
LIMIT 3
""").show()

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          3|     2110.0|
|          2|     1790.0|
|          5|     1650.0|
+-----------+-----------+



pyspark

In [72]:
top3 = customer_spend.limit(3)

top3.show()

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          3|     2110.0|
|          2|     1790.0|
|          5|     1650.0|
+-----------+-----------+



Exercise 3: Customers with No Orders

sql

In [73]:
spark.sql("""
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customers c
LEFT JOIN orders o
ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL
""").show()

+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|         11|       Mia|    Lopez|
|         12|   William| Anderson|
|         13|    Amelia|   Thomas|
|         14|     Ethan|   Taylor|
|         15|    Harper|  Jackson|
|         16|   Jackson|    White|
|         17| Charlotte|   Harris|
|         18|    Oliver|   Martin|
|         19|   Madison| Thompson|
|         20|    Elijah|   Garcia|
|         21|  Scarlett|   Wilson|
|         22|     Henry|    Moore|
|         23|    Evelyn|      Lee|
|         24|    Daniel|   Walker|
|         25|    Harper|     Hall|
|         26|   Matthew|    Allen|
|         27|     Sofia|    Young|
|         28|       Leo|     King|
|         29|      Nora|   Wright|
|         30|     David|    Scott|
+-----------+----------+---------+
only showing top 20 rows


pyspark

In [74]:
customers_no_orders = customers.join(
    orders,
    on="customer_id",
    how="left_anti"
)

customers_no_orders.show()

+-----------+----------+---------+--------------------+------------+--------------+------------+-----+--------+
|customer_id|first_name|last_name|               email|phone_number|       address|        city|state|zip_code|
+-----------+----------+---------+--------------------+------------+--------------+------------+-----+--------+
|         11|       Mia|    Lopez|  mia.lopez@mail.com|    555-0011|    808 Oak St|       Miami|   FL|   33101|
|         12|   William| Anderson|william.anderson@...|    555-0012|   909 Pine St|   Nashville|   TN|   37201|
|         13|    Amelia|   Thomas|amelia.thomas@pro...|    555-0013|   1010 Elm St|      Denver|   CO|   80201|
|         14|     Ethan|   Taylor|ethan.taylor@inbo...|    555-0014| 1111 Birch St| Minneapolis|   MN|   55401|
|         15|    Harper|  Jackson|harper.jackson@ou...|    555-0015| 1212 Cedar St|     Seattle|   WA|   98101|
|         16|   Jackson|    White|jackson.white@yma...|    555-0016|1313 Spruce St|     Atlanta|   GA|  

Exercise 4: City-wise Total Revenue

SQL

In [79]:
orders.createOrReplaceTempView("orders")

In [80]:
spark.sql("""
SELECT
    c.city,
    SUM(o.order_amount) AS total_revenue
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.city
ORDER BY total_revenue DESC
""").show()

+-----------+-------------+
|       city|total_revenue|
+-----------+-------------+
| Greenville|       2110.0|
|Centerville|       1790.0|
|   Lakeside|       1650.0|
|  Riverside|       1550.0|
|Springfield|       1170.0|
|      Boise|        970.0|
| Des Moines|        900.0|
|     Albany|        630.0|
|   Portland|        600.0|
|    Oakland|        585.0|
+-----------+-------------+



PySpark

In [81]:
from pyspark.sql.functions import sum, col

joined_df = customers.join(orders, "customer_id")

city_revenue = joined_df.groupBy("city") \
    .agg(sum("order_amount").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

city_revenue.show()

+-----------+-------------+
|       city|total_revenue|
+-----------+-------------+
| Greenville|       2110.0|
|Centerville|       1790.0|
|   Lakeside|       1650.0|
|  Riverside|       1550.0|
|Springfield|       1170.0|
|      Boise|        970.0|
| Des Moines|        900.0|
|     Albany|        630.0|
|   Portland|        600.0|
|    Oakland|        585.0|
+-----------+-------------+



Exercise 5: Average Order Amount Per Customer
#####SQL

In [82]:
spark.sql("""
SELECT
    customer_id,
    AVG(order_amount) AS average_order_amount
FROM orders
GROUP BY customer_id
ORDER BY average_order_amount DESC
""").show()

+-----------+--------------------+
|customer_id|average_order_amount|
+-----------+--------------------+
|          3|              1055.0|
|          7|               970.0|
|          8|               900.0|
|          2|               895.0|
|          5|               825.0|
|          4|               775.0|
|          9|               630.0|
|         10|               600.0|
|          1|               585.0|
|          6|               585.0|
+-----------+--------------------+



PySpark

In [83]:
from pyspark.sql.functions import avg, col

avg_order = orders.groupBy("customer_id") \
    .agg(
        avg("order_amount").alias("average_order_amount")
    ) \
    .orderBy(col("average_order_amount").desc())

avg_order.show()

+-----------+--------------------+
|customer_id|average_order_amount|
+-----------+--------------------+
|          3|              1055.0|
|          7|               970.0|
|          8|               900.0|
|          2|               895.0|
|          5|               825.0|
|          4|               775.0|
|          9|               630.0|
|         10|               600.0|
|          1|               585.0|
|          6|               585.0|
+-----------+--------------------+



Exercise 6: Customers with More Than One Order

SQL

In [84]:
spark.sql("""
SELECT
    customer_id,
    COUNT(order_id) AS order_count
FROM orders
GROUP BY customer_id
HAVING COUNT(order_id) > 1
ORDER BY order_count DESC
""").show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|          1|          2|
|          3|          2|
|          5|          2|
|          4|          2|
|          2|          2|
+-----------+-----------+



PySpark

In [85]:
from pyspark.sql.functions import count, col

repeat_customers = orders.groupBy("customer_id") \
    .agg(count("order_id").alias("order_count")) \
    .filter(col("order_count") > 1) \
    .orderBy(col("order_count").desc())

repeat_customers.show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|          1|          2|
|          3|          2|
|          5|          2|
|          4|          2|
|          2|          2|
+-----------+-----------+



Exercise 7: Sort Customers by Total Spend (Descending)

SQL

In [86]:
spark.sql("""
SELECT
    customer_id,
    SUM(order_amount) AS total_spend
FROM orders
GROUP BY customer_id
ORDER BY total_spend DESC
""").show()

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          3|     2110.0|
|          2|     1790.0|
|          5|     1650.0|
|          4|     1550.0|
|          1|     1170.0|
|          7|      970.0|
|          8|      900.0|
|          9|      630.0|
|         10|      600.0|
|          6|      585.0|
+-----------+-----------+



PySpark

In [87]:
from pyspark.sql.functions import sum, col

customer_spend = orders.groupBy("customer_id") \
    .agg(
        sum("order_amount").alias("total_spend")
    ) \
    .orderBy(col("total_spend").desc())

customer_spend.show()

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          3|     2110.0|
|          2|     1790.0|
|          5|     1650.0|
|          4|     1550.0|
|          1|     1170.0|
|          7|      970.0|
|          8|      900.0|
|          9|      630.0|
|         10|      600.0|
|          6|      585.0|
+-----------+-----------+

